# Clustering Climate Stations with Machine Learning

Can a neural network learn to group cities by climate type **without being told the answer**? In this notebook, we'll:

1. Fetch temperature data for **20 US stations** spanning diverse climates
2. Extract a **climate fingerprint** (26 numbers) for each station
3. Train a **PyTorch autoencoder** to compress those 26 numbers down to just 2
4. Visualize the results as a **2D scatter plot** where similar climates cluster together

This is the same technique used in the companion Birdsong project to cluster bird species by their song patterns.

## Step 1: Import Libraries

In [ ]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler

## Step 2: Define Our Stations

20 stations across the US covering tropical, subtropical, continental, Mediterranean, marine, arid, and subarctic climates.

In [ ]:
station_list = [
    ("Miami",          "USW00012839", "Tropical"),
    ("New York",       "USW00094728", "Continental"),
    ("Chicago",        "USW00094846", "Continental"),
    ("Los Angeles",    "USW00023174", "Mediterranean"),
    ("Seattle",        "USW00024233", "Marine"),
    ("Phoenix",        "USW00023183", "Arid"),
    ("Denver",         "USW00003017", "Semi-Arid"),
    ("Anchorage",      "USW00026451", "Subarctic"),
    ("Honolulu",       "USW00022521", "Tropical"),
    ("Minneapolis",    "USW00014922", "Continental"),
    ("Atlanta",        "USW00013874", "Subtropical"),
    ("Dallas",         "USW00013960", "Subtropical"),
    ("San Francisco",  "USW00023234", "Mediterranean"),
    ("Boston",         "USW00014739", "Continental"),
    ("New Orleans",    "USW00012916", "Subtropical"),
    ("Las Vegas",      "USW00023169", "Arid"),
    ("Portland",       "USW00024229", "Marine"),
    ("Salt Lake City", "USW00024127", "Semi-Arid"),
    ("Jacksonville",   "USW00013889", "Subtropical"),
    ("Nashville",      "USW00013897", "Subtropical"),
]

print(f"Total stations: {len(station_list)}")
for name, sid, zone in station_list:
    print(f"  {name:18s} {sid}  ({zone})")

## Step 3: Fetch and Clean Data

In [ ]:
def fetch_station(station_id, name):
    """Fetch daily TAVG/TMAX/TMIN from NOAA and return a cleaned DataFrame."""
    url = "https://www.ncei.noaa.gov/access/services/data/v1"
    params = {
        "dataset": "daily-summaries",
        "stations": station_id,
        "startDate": "2000-01-01",
        "endDate": "2024-12-31",
        "dataTypes": "TMAX,TMIN,TAVG",
        "units": "metric",
        "format": "json"
    }
    r = requests.get(url, params=params, timeout=60)
    df = pd.DataFrame(r.json())
    df["DATE"] = pd.to_datetime(df["DATE"])
    for col in ["TAVG", "TMAX", "TMIN"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df["TAVG"] = df["TAVG"].fillna((df["TMAX"] + df["TMIN"]) / 2)
    df["year"] = df["DATE"].dt.year
    df["month"] = df["DATE"].dt.month
    return df

# Fetch all stations (this takes a couple of minutes)
print("Fetching 20 stations from NOAA... (this may take a few minutes)")
all_data = {}
for i, (name, sid, zone) in enumerate(station_list):
    print(f"  [{i+1}/{len(station_list)}] {name}...", end=" ")
    try:
        all_data[name] = fetch_station(sid, name)
        print(f"{len(all_data[name])} records")
    except Exception as e:
        print(f"FAILED: {e}")

print(f"\nSuccessfully fetched: {len(all_data)}/{len(station_list)} stations")

## Step 4: Feature Extraction — Climate "Fingerprint"

In the Birdsong project, we used **MFCCs** — 13 numbers describing the frequency shape of a recording (plus 13 standard deviations = 26 features total).

Here, we compute a **climate fingerprint** with the same structure:
- **12 monthly mean temperatures** (the "shape" of the seasonal cycle)
- **12 monthly standard deviations** (how variable each month is)
- **1 annual range** (max monthly mean − min monthly mean)
- **1 trend slope** (warming or cooling rate)

**Total: 26 features per station** — matching the Birdsong project exactly.

In [ ]:
def extract_features(df):
    """Extract 26 climate features from a station's daily data."""
    monthly = df.groupby("month")["TAVG"]

    monthly_means = monthly.mean().values           # 12 features
    monthly_stds = monthly.std().values             # 12 features
    annual_range = monthly_means.max() - monthly_means.min()  # 1 feature

    # Trend: slope of annual averages over time
    annual = df.groupby("year")["TAVG"].mean()
    if len(annual) > 1:
        slope = np.polyfit(annual.index, annual.values, 1)[0]
    else:
        slope = 0.0

    return np.concatenate([monthly_means, monthly_stds, [annual_range, slope]])

# Extract features for all stations
feature_names = (
    [f"mean_{m}" for m in range(1, 13)] +
    [f"std_{m}" for m in range(1, 13)] +
    ["annual_range", "trend_slope"]
)

features = {}
for name in all_data:
    features[name] = extract_features(all_data[name])

feature_df = pd.DataFrame(features, index=feature_names).T
print(f"Feature matrix: {feature_df.shape[0]} stations × {feature_df.shape[1]} features")
feature_df.head(10)

## Step 5: Preprocessing

Normalize all features to mean=0, standard deviation=1. Without this, features with large values (like monthly means in °C) would dominate training over features with small values (like trend slopes).

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(feature_df.values)

# Convert to PyTorch tensor
X_tensor = torch.FloatTensor(X_scaled)
print(f"Tensor shape: {X_tensor.shape}")
print(f"Feature means (should be ~0): {X_scaled.mean(axis=0)[:3].round(4)}")
print(f"Feature stds (should be ~1):  {X_scaled.std(axis=0)[:3].round(4)}")

## Step 6: The Autoencoder — Compressing Climate to 2D

An **autoencoder** is a neural network that learns to compress data:

```
Input (26 features) → Encoder → Bottleneck (2 numbers) → Decoder → Output (26 features)
```

The network is trained to reconstruct its own input. The **bottleneck** forces it to find the 2 most important dimensions. If reconstruction is good, those 2 numbers capture the essence of the climate.

This is the **same architecture** used in the Birdsong project for clustering bird species.

In [ ]:
class ClimateAutoencoder(nn.Module):
    def __init__(self, input_dim=26, bottleneck=2):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, bottleneck),
        )
        self.decoder = nn.Sequential(
            nn.Linear(bottleneck, 64),
            nn.ReLU(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, input_dim),
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)

model = ClimateAutoencoder(input_dim=26, bottleneck=2)
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

## Step 7: Train the Autoencoder

In [ ]:
LEARNING_RATE = 0.0001
EPOCHS = 5000

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

losses = []
for epoch in range(EPOCHS):
    output = model(X_tensor)
    loss = criterion(output, X_tensor)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 100 == 0:
        losses.append(loss.item())
    if epoch % 1000 == 0:
        print(f"  Epoch {epoch:5d}: loss = {loss.item():.6f}")

print(f"  Final loss: {loss.item():.6f}")

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(range(0, EPOCHS, 100), losses, color="steelblue")
plt.title("Autoencoder Training Loss")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.grid(True, alpha=0.3)
plt.show()

print("Lower loss = better compression.")
print("The network learned which 2 numbers best summarize 26 climate features.")

## Step 8: Visualize the Clusters

Extract the 2D bottleneck values and plot them. Each point is a city. Color by climate zone.

In [ ]:
# Extract 2D embeddings
model.eval()
with torch.no_grad():
    embeddings = model.encoder(X_tensor).numpy()

# Build plot data
zone_lookup = {name: zone for name, _, zone in station_list}
zone_colors = {
    "Tropical": "#e74c3c",
    "Subtropical": "#e67e22",
    "Continental": "#3498db",
    "Mediterranean": "#2ecc71",
    "Marine": "#9b59b6",
    "Arid": "#f1c40f",
    "Semi-Arid": "#d4ac0d",
    "Subarctic": "#1abc9c",
}

plt.figure(figsize=(12, 8))

for i, name in enumerate(feature_df.index):
    zone = zone_lookup.get(name, "Unknown")
    color = zone_colors.get(zone, "gray")
    plt.scatter(embeddings[i, 0], embeddings[i, 1], c=color, s=100, zorder=3, edgecolors="black", linewidth=0.5)
    plt.annotate(name, (embeddings[i, 0], embeddings[i, 1]),
                 textcoords="offset points", xytext=(6, 6), fontsize=8)

# Legend
for zone, color in zone_colors.items():
    plt.scatter([], [], c=color, s=80, label=zone, edgecolors="black", linewidth=0.5)
plt.legend(title="Climate Zone", loc="best")

plt.title("Climate Station Clustering (Autoencoder 2D Embedding)")
plt.xlabel("Dimension 1")
plt.ylabel("Dimension 2")
plt.grid(True, alpha=0.3)
plt.show()

## Step 9: Interpret the Results

Look at your scatter plot and consider:

- **Which cities cluster together?** Do nearby points share similar climates?
- **Does geography predict clustering?** Are coastal cities together? Northern cities?
- **Any surprises?** Did any city end up in an unexpected location?
- **Tight vs. loose clusters:** Tight groups = very similar climates. Spread-out groups = more variation within that climate type.

The autoencoder discovered these groupings **entirely on its own** — it was never told the climate zone labels!

## Step 10: Experiment!

Try changing these hyperparameters and re-running the training + plotting cells:

| Parameter | Default | Try |
|-----------|---------|-----|
| `LEARNING_RATE` | 0.0001 | 0.001, 0.00001 |
| `EPOCHS` | 5000 | 500, 10000 |
| `bottleneck` | 2 | 3, 5 (still plot first 2 dims) |

Questions to explore:
- What happens with too few epochs? (underfitting)
- What happens with a very high learning rate? (instability)
- Does a larger bottleneck produce better clusters?

In [ ]:
# === EXPERIMENT: Change these and re-run ===
LEARNING_RATE = 0.0001   # Try: 0.001, 0.00001
EPOCHS = 5000            # Try: 500, 10000
BOTTLENECK = 2           # Try: 3, 5

model_exp = ClimateAutoencoder(input_dim=26, bottleneck=BOTTLENECK)
criterion_exp = nn.MSELoss()
optimizer_exp = torch.optim.Adam(model_exp.parameters(), lr=LEARNING_RATE)

for epoch in range(EPOCHS):
    output = model_exp(X_tensor)
    loss = criterion_exp(output, X_tensor)
    optimizer_exp.zero_grad()
    loss.backward()
    optimizer_exp.step()

print(f"Final loss: {loss.item():.6f}")

# Plot
model_exp.eval()
with torch.no_grad():
    emb = model_exp.encoder(X_tensor).numpy()

plt.figure(figsize=(12, 8))
for i, name in enumerate(feature_df.index):
    zone = zone_lookup.get(name, "Unknown")
    color = zone_colors.get(zone, "gray")
    plt.scatter(emb[i, 0], emb[i, 1], c=color, s=100, zorder=3, edgecolors="black", linewidth=0.5)
    plt.annotate(name, (emb[i, 0], emb[i, 1]), textcoords="offset points", xytext=(6, 6), fontsize=8)

for zone, color in zone_colors.items():
    plt.scatter([], [], c=color, s=80, label=zone, edgecolors="black", linewidth=0.5)
plt.legend(title="Climate Zone", loc="best")
plt.title(f"Experiment: lr={LEARNING_RATE}, epochs={EPOCHS}, bottleneck={BOTTLENECK}")
plt.xlabel("Dimension 1")
plt.ylabel("Dimension 2")
plt.grid(True, alpha=0.3)
plt.show()

## What's Next

In **Notebook 4**, we'll go beyond station data to explore **satellite imagery** (GOES-19) and **gridded reanalysis datasets** (NCEP/NCAR) — seeing the whole atmosphere at once.